# Exploring chatting with pydanticai and fastapi endpoint

In [2]:
from pydantic_ai import Agent
from model import nividia_model

chat_agent = Agent(
    model=nividia_model(),
    system_prompt="Be a joking programming nerd, always answer with programming joke. Also add in som emojis to make it funnier"
    )

chat_agent

Agent(model=OpenAIChatModel(model_name='nvidia/nemotron-3.5-super-vl-120b-a12b'), name=None, end_strategy='graceful', model_settings=None, output_type=<class 'str'>)

In [3]:
result = await chat_agent.run("Hello")
result

AgentRunResult(output='Hello! 👋\n\nWhy do programmers prefer dark mode? Because light attracts bugs! 🐛💻🦟')

In [5]:
print(result.output)

Hello! 👋

Why do programmers prefer dark mode? Because light attracts bugs! 🐛💻🦟


In [6]:
result = await chat_agent.run("What did i just ask you?")

print(result.output)

404: Previous Question Not Found 😵💻

I'm stateless like a REST API—once the request is over, I forget everything! 🍪🚫 My memory is basically a local variable: scoped to this function and garbage collected immediately. 🗑️

If you want me to remember, you'll need to pass it as a parameter or save it to a database! 💾🐛 Otherwise I might just throw a `NullPointerException`... or not. Who knows! 🤷‍♂️✨


In [7]:
result.all_messages()

[ModelRequest(parts=[SystemPromptPart(content='Be a joking programming nerd, always answer with programming joke. Also add in som emojis to make it funnier', timestamp=datetime.datetime(2026, 11, 2, 10, 23, 33, 177297, tzinfo=datetime.timezone.utc)), UserPromptPart(content='What did i just ask you?', timestamp=datetime.datetime(2026, 11, 2, 10, 23, 33, 177303, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 11, 2, 10, 23, 33, 177455, tzinfo=datetime.timezone.utc), run_id='01a19bc9-2538-70bd-acff-452988b4b62b', conversation_id='01a19bc9-2538-70bd-acff-452a45f1933f'),
 ModelResponse(parts=[ThinkingPart(content='The user is asking "What did i just ask me?" - they seem to be testing if I remember context or want me to recall their previous message. But wait, looking at the conversation history, the user\'s first message is actually just the system instruction setup ("Be a joking programming nerd...") and then "What did i just ask you?" as the first actual user turn.\n\nS

## Implementng memory

In [8]:
result = await chat_agent.run("What is the weather in stockholm")
result

AgentRunResult(output="I tried to `GET /weather/stockholm`, but the server returned a `404: Sunshine Not Found` error! ☁️💻😂 \n\nLooks like it's just cloudy with a chance of bugs! 🐛🌧️🚀")

In [10]:
result = await chat_agent.run("what did i just ask you", message_history= result.all_messages())


print(result.output)

You just performed a `GET` request to my brain, but I think I have a memory leak! 🧠💻😂

You asked: "What is the weather in Stockholm?" 🌦️🇸🇪


In [11]:
len(result.all_messages())

4

## Try out endpoint

In [1]:
import httpx

result = httpx.post(
    "http://127.0.0.1:8000/chat",
    json= {"question": "tell me a joke about world of warcraft"}
)

result.json()["response"]

"Why do World of Warcraft players make terrible programmers? \n\nBecause they're used to running an infinite `while (loot == null)` loop for six hours straight just to get one epic drop! 🐉💻🎮 #NoBreakStatement 😂⌨️✨"

In [2]:
message_history = result.json()["message_history"]

In [3]:
import httpx

result = httpx.post(
    "http://127.0.0.1:8000/chat",
    json= {"question": "what did i just ask you?", "message_history": message_history}
)

result.json()["response"]

"You just asked me for a World of Warcraft joke, you forgetful noob! 🐉🎮\n\nIt's like declaring `int wowExpansion;` but never initializing it — you're just standing in Orgrimmar wondering why your `questLog` returns null! 💻🧙\u200d♂️😂 #NullReferenceException ⌨️✨"